# GenerativeAURORA — Baseline B: Conditional VAE Scenario Generator

### Purpose
This notebook converts the supplied Baseline B Conditional VAE script into notebook format while preserving its implementation exactly.

### Workflow position
**Sequence preparation → Baseline A LSTM → Baseline B CVAE → other baselines → comparison with GenerativeAURORA**

### Architecture in the supplied script
The baseline is a Conditional Variational Autoencoder (CVAE) for solar GHI scenario generation. Weather context is encoded with a bidirectional GRU, while regime and site embeddings condition the latent encoder and decoder. At inference, latent vectors are sampled from a standard normal distribution and decoded into GHI trajectories.

### Training objective
The supplied implementation uses reconstruction MSE plus a KL-divergence term weighted by `BETA_KL = 0.5`.

### Faithful-conversion rule
The attached `step12_Baseline_B_VAE.py` is the source of truth. The Python code below is copied **verbatim** and only divided into logical notebook cells. No architecture, loss, parameter, metric, output filename, or workflow step is silently changed.

The Markdown cells add explanation only.

## 1. Header and imports

This notebook is a faithful conversion of the supplied Baseline B Conditional VAE script. The original executable Python is preserved verbatim; Markdown cells only document the existing implementation.

In [ ]:
# GenerativeAURORA - Baseline B: Conditional VAE Scenario Generator
#
# A Conditional Variational Autoencoder (CVAE) for solar GHI scenario generation.
# This is the standard generative baseline before diffusion models, commonly used
# in energy scenario generation literature (Applied Energy, IEEE TPWRS).
#
# Architecture:
#   Encoder : BiGRU(weather context) -> mu, log_var in latent space (dim=32)
#   Decoder : MLP(z + regime_emb + site_emb) -> GHI trajectory (length 16)
#   Training: ELBO = reconstruction loss + KL divergence (beta-VAE with beta=0.5)
#
# At inference: sample z ~ N(0,I) and decode -> scenario
#
# Run:
#   python baseline_b_vae.py

import os
import sys
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Paths

Defines the original dataset, evaluation, and checkpoint directories. The Windows paths are intentionally preserved.

In [ ]:
# -----------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------
DATASET_DIR = "/content/drive/MyDrive/Colab Notebooks/Dataset"
EVAL_DIR    = os.path.join(DATASET_DIR, "evaluation")
CKPT_DIR    = os.path.join(DATASET_DIR, "checkpoints_vae")
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

## 3. Hyperparameters

Preserves the original sequence dimensions, latent dimension, embedding size, hidden size, batch size, epochs, optimizer settings, beta-VAE coefficient, gradient clipping, and number of scenarios.

In [ ]:
# -----------------------------------------------------------------------
# HYPERPARAMETERS
# -----------------------------------------------------------------------
HORIZON     = 16
LOOKBACK    = 16
N_FEATURES  = 10
LATENT_DIM  = 32
EMBED_DIM   = 32
HIDDEN_DIM  = 128
BATCH_SIZE  = 128
EPOCHS      = 150
LR          = 1e-3
WEIGHT_DECAY= 1e-5
BETA_KL     = 0.5        # beta-VAE coefficient: balance reconstruction vs KL
GRAD_CLIP   = 1.0
N_SCENARIOS = 50

REGIME_NAMES = {0: "Clear", 1: "Cloudy", 2: "Overcast"}
SITE_NAMES   = {0: "Phoenix_AZ", 1: "LosAngeles_CA", 2: "Denver_CO",
                3: "Miami_FL",   4: "Seattle_WA"}

## 4. Dataset

Loads the same sequence arrays used by the main GenerativeAURORA pipeline, including conditioning features, GHI targets, sky regimes, and site labels.

In [ ]:
# -----------------------------------------------------------------------
# DATASET
# -----------------------------------------------------------------------
class SolarSequenceDataset(Dataset):
    def __init__(self, npz_path):
        data        = np.load(npz_path)
        self.X      = torch.tensor(data["X"],      dtype=torch.float32)
        self.Y      = torch.tensor(data["Y"],      dtype=torch.float32)
        self.regime = torch.tensor(data["regime"], dtype=torch.long)
        self.site   = torch.tensor(data["site"],   dtype=torch.long)

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.regime[idx], self.site[idx]

## 5. Weighted sampler

Creates the inverse-frequency weighted sampler used during training, matching the source script.

In [ ]:
def make_weighted_sampler(dataset):
    labels = dataset.regime.numpy()
    unique, counts = np.unique(labels, return_counts=True)
    freq    = dict(zip(unique.tolist(), counts.tolist()))
    weights = np.array([1.0 / freq[int(r)] for r in labels], dtype=np.float32)
    return WeightedRandomSampler(torch.from_numpy(weights), len(weights), replacement=True)

## 6. Conditional VAE model

Defines the Conditional VAE exactly as supplied: a bidirectional GRU context encoder, GHI trajectory encoder, conditional latent encoder, MLP decoder, reparameterization, ELBO loss, and stochastic scenario sampling.

In [ ]:
# -----------------------------------------------------------------------
# MODEL: Conditional VAE
# -----------------------------------------------------------------------
class ConditionEncoder(nn.Module):
    """Encodes the 16x10 weather context into a fixed vector."""
    def __init__(self, n_features=N_FEATURES, hidden=HIDDEN_DIM, embed_dim=EMBED_DIM):
        super().__init__()
        self.gru  = nn.GRU(n_features, hidden, num_layers=2,
                           batch_first=True, bidirectional=True, dropout=0.1)
        self.proj = nn.Linear(hidden * 2, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        # x: (B, L, F)
        _, h = self.gru(x)          # h: (4, B, hidden) -- 2 layers x bidirectional
        h    = torch.cat([h[-2], h[-1]], dim=-1)   # (B, hidden*2)
        return self.norm(self.proj(h))              # (B, embed_dim)


class VAEEncoder(nn.Module):
    """Encodes (Y, context) -> (mu, log_var) in latent space."""
    def __init__(self, horizon=HORIZON, embed_dim=EMBED_DIM,
                 latent_dim=LATENT_DIM, hidden=HIDDEN_DIM):
        super().__init__()
        # Y encoder: simple MLP on the GHI trajectory
        self.y_enc = nn.Sequential(
            nn.Linear(horizon, hidden),
            nn.SiLU(),
            nn.Linear(hidden, embed_dim),
        )
        # Fuse Y encoding with context
        self.fuse  = nn.Sequential(
            nn.Linear(embed_dim * 3, hidden),   # y + context + regime+site
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
        )
        self.mu_head      = nn.Linear(hidden, latent_dim)
        self.logvar_head  = nn.Linear(hidden, latent_dim)

    def forward(self, y, ctx, regime_e, site_e):
        y_e  = self.y_enc(y)
        fused = self.fuse(torch.cat([y_e, ctx, regime_e + site_e], dim=-1))
        return self.mu_head(fused), self.logvar_head(fused)


class VAEDecoder(nn.Module):
    """Decodes (z, context, regime, site) -> GHI trajectory."""
    def __init__(self, horizon=HORIZON, embed_dim=EMBED_DIM,
                 latent_dim=LATENT_DIM, hidden=HIDDEN_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + embed_dim * 3, hidden * 2),
            nn.SiLU(),
            nn.Linear(hidden * 2, hidden * 2),
            nn.SiLU(),
            nn.Linear(hidden * 2, hidden),
            nn.SiLU(),
            nn.Linear(hidden, horizon),
            nn.Sigmoid(),    # output in [0,1] normalized scale
        )

    def forward(self, z, ctx, regime_e, site_e):
        inp = torch.cat([z, ctx, regime_e, site_e], dim=-1)
        return self.net(inp)


class ConditionalVAE(nn.Module):
    def __init__(self, n_regimes=3, n_sites=5,
                 embed_dim=EMBED_DIM, latent_dim=LATENT_DIM):
        super().__init__()
        self.regime_emb = nn.Embedding(n_regimes, embed_dim)
        self.site_emb   = nn.Embedding(n_sites,   embed_dim)
        self.ctx_enc    = ConditionEncoder(embed_dim=embed_dim)
        self.vae_enc    = VAEEncoder(embed_dim=embed_dim, latent_dim=latent_dim)
        self.vae_dec    = VAEDecoder(embed_dim=embed_dim, latent_dim=latent_dim)
        self.latent_dim = latent_dim

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x, y, regime, site):
        # Encode conditioning
        r_e = self.regime_emb(regime)
        s_e = self.site_emb(site)
        ctx = self.ctx_enc(x)

        # Encode posterior
        mu, log_var = self.vae_enc(y, ctx, r_e, s_e)
        z           = self.reparameterize(mu, log_var)

        # Decode
        y_hat = self.vae_dec(z, ctx, r_e, s_e)

        # ELBO
        recon_loss = F.mse_loss(y_hat, y)
        kl_loss    = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
        loss       = recon_loss + BETA_KL * kl_loss
        return loss, recon_loss.item(), kl_loss.item()

    @torch.no_grad()
    def sample(self, x, regime, site, n_scenarios=N_SCENARIOS):
        # x      : (B, L, F)
        # Returns: (B, n_scenarios, H)
        B      = x.shape[0]
        device = x.device

        r_e = self.regime_emb(regime)
        s_e = self.site_emb(site)
        ctx = self.ctx_enc(x)

        # Expand for scenarios
        ctx = ctx.unsqueeze(1).expand(-1, n_scenarios, -1).reshape(B * n_scenarios, -1)
        r_e = r_e.unsqueeze(1).expand(-1, n_scenarios, -1).reshape(B * n_scenarios, -1)
        s_e = s_e.unsqueeze(1).expand(-1, n_scenarios, -1).reshape(B * n_scenarios, -1)

        z     = torch.randn(B * n_scenarios, self.latent_dim, device=device)
        y_hat = self.vae_dec(z, ctx, r_e, s_e)   # (B*S, H)
        return y_hat.reshape(B, n_scenarios, -1)   # (B, S, H)

## 7. Evaluation metrics

Implements the original W/m²-scale metrics: CRPS, pinball loss, prediction-interval coverage/width, median MAE, and the printed metric summary.

In [ ]:
# -----------------------------------------------------------------------
# METRICS (identical to step6_evaluate.py)
# -----------------------------------------------------------------------
def load_scalers():
    with open(os.path.join(DATASET_DIR, "scaler_params.json"), "r") as f:
        return json.load(f)

def inv_scale_ghi(arr, scalers):
    s = scalers["GHI"]
    return arr * (s["max"] - s["min"]) + s["min"]

def crps_score(actuals, scenarios):
    N, S, H = scenarios.shape
    t1 = np.abs(scenarios - actuals[:, np.newaxis, :]).mean(axis=1)
    t2 = np.zeros((N, H))
    for n in range(N):
        sc = scenarios[n]
        t2[n] = np.abs(sc[:, np.newaxis, :] - sc[np.newaxis, :, :]).mean(axis=(0,1))
    return float((t1 - 0.5 * t2).mean())

def pinball_np(actuals, scenarios, q):
    fc = np.quantile(scenarios, q, axis=1)
    e  = actuals - fc
    return float(np.where(e >= 0, q*e, (q-1)*e).mean())

def coverage_width(actuals, scenarios, alpha=0.10):
    lo  = np.quantile(scenarios, alpha/2,   axis=1)
    hi  = np.quantile(scenarios, 1-alpha/2, axis=1)
    cov = float(((actuals >= lo) & (actuals <= hi)).mean()) * 100
    wid = float((hi - lo).mean())
    return round(cov, 4), round(wid, 4)

def med_mae(actuals, scenarios):
    return float(np.abs(np.median(scenarios, axis=1) - actuals).mean())

def print_metrics(actuals, scenarios, label=""):
    crps = crps_score(actuals, scenarios)
    pb10 = pinball_np(actuals, scenarios, 0.10)
    pb50 = pinball_np(actuals, scenarios, 0.50)
    pb90 = pinball_np(actuals, scenarios, 0.90)
    cov90, w90 = coverage_width(actuals, scenarios, 0.10)
    cov80, _   = coverage_width(actuals, scenarios, 0.20)
    mae  = med_mae(actuals, scenarios)
    if label: print("  " + label)
    print("    CRPS         : " + str(round(crps, 4)))
    print("    Pinball p10  : " + str(round(pb10, 4)))
    print("    Pinball p50  : " + str(round(pb50, 4)))
    print("    Pinball p90  : " + str(round(pb90, 4)))
    print("    Coverage 90% : " + str(cov90) + "%")
    print("    Coverage 80% : " + str(cov80) + "%")
    print("    Width 90% PI : " + str(w90) + " W/m2")
    print("    Median MAE   : " + str(round(mae, 4)))
    return {"crps": crps, "pinball_p10": pb10, "pinball_p50": pb50,
            "pinball_p90": pb90, "coverage_90": cov90, "coverage_80": cov80,
            "width_90": w90, "median_mae": mae}

## 8. Main setup and data loading

Initializes the device, datasets, data loaders, Conditional VAE, optimizer, scheduler, checkpoint path, and training log exactly as in the source.

In [ ]:
# -----------------------------------------------------------------------
# MAIN
# -----------------------------------------------------------------------
def main():
    print("GenerativeAURORA - Baseline B: Conditional VAE")
    print("")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device : " + str(device))
    print("")

    train_ds = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_train.npz"))
    val_ds   = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_val.npz"))
    test_ds  = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_test.npz"))
    print("Train: " + str(len(train_ds)) + "  Val: " + str(len(val_ds)) + "  Test: " + str(len(test_ds)))

    sampler      = make_weighted_sampler(train_ds)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=0, pin_memory=(device.type=="cuda"))
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,    num_workers=0, pin_memory=(device.type=="cuda"))
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,    num_workers=0)

    model = ConditionalVAE().to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print("Parameters: " + str(total_params))
    print("")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    best_val  = float("inf")
    best_path = os.path.join(CKPT_DIR, "vae_best.pt")
    log_path  = os.path.join(DATASET_DIR, "training_log_vae.csv")

## 9. Training and validation loop

Runs the original training and validation loops for the configured number of epochs, saves the best validation checkpoint, and reports training time and best validation loss.

In [ ]:
    print("Training for " + str(EPOCHS) + " epochs...")
    print("-" * 60)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ConditionalVAE().to(device)
    train_ds = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_train.npz"))
    val_ds   = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_val.npz"))
    test_ds  = SolarSequenceDataset(os.path.join(DATASET_DIR, "sequences_test.npz"))
    sampler      = make_weighted_sampler(train_ds)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=0, pin_memory=(device.type=="cuda"))
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,    num_workers=0, pin_memory=(device.type=="cuda"))
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,    num_workers=0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    log_path  = os.path.join(DATASET_DIR, "training_log_vae.csv")
    best_val  = float("inf")

    with open(log_path, "w") as f:
        f.write("epoch,train_loss,val_loss\n")

    t0 = time.time()
    for epoch in range(EPOCHS):
        # -- Train --
        model.train()
        tr_total = 0.0; n = 0
        for X, Y, regime, site in train_loader:
            X, Y = X.to(device), Y.to(device)
            regime, site = regime.to(device), site.to(device)
            optimizer.zero_grad()
            loss, _, _ = model(X, Y, regime, site)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            tr_total += loss.item(); n += 1
        tr = tr_total / n

        # -- Validate --
        model.eval()
        vl_total = 0.0; n = 0
        with torch.no_grad():
            for X, Y, regime, site in val_loader:
                X, Y = X.to(device), Y.to(device)
                regime, site = regime.to(device), site.to(device)
                loss, _, _ = model(X, Y, regime, site)
                vl_total += loss.item(); n += 1
        vl = vl_total / n

        scheduler.step()
        is_best = vl < best_val
        if is_best:
            best_val = vl
            torch.save(model.state_dict(), best_path)

        with open(log_path, "a") as f:
            f.write(str(epoch+1) + "," + str(round(tr,6)) + "," + str(round(vl,6)) + "\n")

        if (epoch + 1) % 10 == 0 or is_best:
            marker = "  <-- best" if is_best else ""
            print("Epoch " + str(epoch+1).rjust(3) + "/" + str(EPOCHS)
                  + "  train: " + str(round(tr,6)).ljust(10)
                  + "  val: "   + str(round(vl,6)).ljust(10) + marker)

    elapsed = round((time.time() - t0) / 60, 1)
    print("-" * 60)
    print("Training complete. Time: " + str(elapsed) + " min")
    print("Best val loss: " + str(round(best_val, 6)))
    print("")

Training for 150 epochs...
------------------------------------------------------------


/tmp/ipykernel_4172/2789641627.py:48: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch   1/150  train: 0.035558    val: 0.013929    <-- best
Epoch   2/150  train: 0.005089    val: 0.007802    <-- best
Epoch   3/150  train: 0.003385    val: 0.005178    <-- best
Epoch   4/150  train: 0.002579    val: 0.003893    <-- best
Epoch   5/150  train: 0.002101    val: 0.003604    <-- best
Epoch   6/150  train: 0.001847    val: 0.003082    <-- best
Epoch   7/150  train: 0.00171     val: 0.003001    <-- best
Epoch   8/150  train: 0.001764    val: 0.00281     <-- best
Epoch   9/150  train: 0.00171     val: 0.002659    <-- best
Epoch  10/150  train: 0.001604    val: 0.002683  
Epoch  11/150  train: 0.001491    val: 0.002419    <-- best
Epoch  12/150  train: 0.001287    val: 0.0022      <-- best
Epoch  14/150  train: 0.001187    val: 0.001943    <-- best
Epoch  15/150  train: 0.001099    val: 0.001899    <-- best
Epoch  16/150  train: 0.001046    val: 0.001801    <-- best
Epoch  17/150  train: 0.001028    val: 0.001697    <-- best
Epoch  19/150  train: 0.000954    val: 0.001651   

## 10. Test-set scenario generation

Loads the best VAE, generates 50 scenarios per test sequence, inverse-scales actuals and scenarios, and computes overall, regime-level, and site-level metrics.

In [ ]:
    CKPT_DIR    = os.path.join(DATASET_DIR, "checkpoints_vae")
    best_path = os.path.join(CKPT_DIR, "vae_best.pt")

    # Evaluation
    print("Evaluating on test set...")
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()
    scalers = load_scalers()

    all_sc, all_act, all_reg, all_sit = [], [], [], []
    with torch.no_grad():
        for X, Y, regime, site in test_loader:
            X = X.to(device); regime = regime.to(device); site = site.to(device)
            sc = model.sample(X, regime, site, N_SCENARIOS).cpu().numpy()
            all_sc.append(sc)
            all_act.append(Y.numpy())
            all_reg.append(regime.cpu().numpy())
            all_sit.append(site.cpu().numpy())

    scenarios = np.concatenate(all_sc,  axis=0)
    actuals   = np.concatenate(all_act, axis=0)
    regimes   = np.concatenate(all_reg, axis=0)
    sites     = np.concatenate(all_sit, axis=0)

    actuals_wm2   = inv_scale_ghi(actuals,   scalers)
    scenarios_wm2 = inv_scale_ghi(scenarios, scalers)

    print("")
    print("=== Conditional VAE: Test Metrics ===")
    overall = print_metrics(actuals_wm2, scenarios_wm2, "Overall (920 days, 5 sites)")

    print("")
    print("=== By Sky Regime ===")
    regime_metrics = {}
    for rid, rname in REGIME_NAMES.items():
        mask = (regimes == rid)
        if mask.sum() == 0: continue
        print("  " + rname + " (n=" + str(mask.sum()) + ")")
        regime_metrics[rname] = print_metrics(actuals_wm2[mask], scenarios_wm2[mask])

    print("")
    print("=== By Site ===")
    site_metrics = {}
    for sid, sname in SITE_NAMES.items():
        mask = (sites == sid)
        if mask.sum() == 0: continue
        print("  " + sname + " (n=" + str(mask.sum()) + ")")
        site_metrics[sname] = print_metrics(actuals_wm2[mask], scenarios_wm2[mask])

Evaluating on test set...

=== Conditional VAE: Test Metrics ===
  Overall (920 days, 5 sites)
    CRPS         : 8.9589
    Pinball p10  : 4.2242
    Pinball p50  : 4.9593
    Pinball p90  : 3.9538
    Coverage 90% : 17.8329%
    Coverage 80% : 14.3886%
    Width 90% PI : 6.3125 W/m2
    Median MAE   : 9.9186

=== By Sky Regime ===
  Clear (n=872)
    CRPS         : 8.9993
    Pinball p10  : 4.2376
    Pinball p50  : 4.9933
    Pinball p90  : 3.9577
    Coverage 90% : 18.3845%
    Coverage 80% : 14.8151%
    Width 90% PI : 6.5017 W/m2
    Median MAE   : 9.9865
  Cloudy (n=46)
    CRPS         : 8.2612
    Pinball p10  : 3.9596
    Pinball p50  : 4.3643
    Pinball p90  : 3.9339
    Coverage 90% : 7.7446%
    Coverage 80% : 6.5217%
    Width 90% PI : 2.941 W/m2
    Median MAE   : 8.7286
  Overcast (n=2)
    CRPS         : 7.3998
    Pinball p10  : 4.4886
    Pinball p50  : 3.8264
    Pinball p90  : 2.7284
    Coverage 90% : 9.375%
    Coverage 80% : 9.375%
    Width 90% PI : 1.3635 W/m

## 11. Save evaluation results

Saves the VAE scenario arrays and JSON metrics to the original evaluation directory.

In [ ]:
    # Save
    np.savez_compressed(
        os.path.join(EVAL_DIR, "vae_results.npz"),
        actuals=actuals_wm2, scenarios=scenarios_wm2,
        regimes=regimes, sites=sites,
    )
    with open(os.path.join(EVAL_DIR, "vae_metrics.json"), "w") as f:
        json.dump({"model": "ConditionalVAE", "overall": overall,
                   "by_regime": regime_metrics, "by_site": site_metrics}, f, indent=2)

    print("")
    print("=== COPY THESE INTO TABLE II ===")
    print("Conditional VAE  |  CRPS: " + str(round(overall["crps"],2))
          + "  | Pb p10: " + str(round(overall["pinball_p10"],2))
          + "  | Pb p50: " + str(round(overall["pinball_p50"],2))
          + "  | Cov 90%: " + str(overall["coverage_90"])
          + "%  | Width: " + str(round(overall["width_90"],1))
          + "  | MAE: " + str(round(overall["median_mae"],2)))
    print("")
    print("Results saved to: " + EVAL_DIR)
    print("Next: run baseline_c_timegrad.py")


=== COPY THESE INTO TABLE II ===
Conditional VAE  |  CRPS: 8.96  | Pb p10: 4.22  | Pb p50: 4.96  | Cov 90%: 17.8329%  | Width: 6.3  | MAE: 9.92

Results saved to: /content/drive/MyDrive/Colab Notebooks/Dataset/evaluation
Next: run baseline_c_timegrad.py


## Execution note

The original Windows paths are intentionally preserved:

`C:/Users/user/Paper_10/Dataset`

`C:/Users/user/Paper_10/Coding`

For actual Colab execution, these paths must point to the corresponding dataset and source-code locations.

### Main inputs
- `sequences_train.npz`
- `sequences_val.npz`
- `sequences_test.npz`
- `scaler_params.json`

### Main outputs
- `evaluation/vae_results.npz`
- `evaluation/vae_metrics.json`
- `training_log_vae.csv`
- `checkpoints_vae/vae_best.pt`

The source script ends by directing the workflow to `baseline_c_timegrad.py`.